In [40]:
!pip install -U langchain langchain-core langchain-community langchain-deepseek

In [3]:
import os
from langchain_deepseek import ChatDeepSeek
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser,JsonOutputParser
import glob


c:\Users\wei10\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
#Initialize the Model with api_key
llm = ChatDeepSeek(
    model="deepseek-chat", 
    temperature=0.4, 
    api_key="sk-b03ebd023fa744daa62e8d2c0d899111" 
)
# Initialize an empty conversation history.
chat_history = []


#RAG

In [ ]:
#Execute python scripts/preprocess.py and python scripts/index.py in terminal before executing this section
def get_retriever():
    #Perform embedding
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = FAISS.load_local("vector", embeddings, allow_dangerous_deserialization=True)  
    return vectorstore.as_retriever(search_kwargs={"k": 3})

retriever = get_retriever()

In [ ]:
classifier_prompt = ChatPromptTemplate.from_template(
    "Based on the student question. Detect the subject (Math, Science, Social Studies or English) "
    "and the grade level (1, 2, 3, 4, or 5).\n"
    "Return JSON only: {{\"subject\": \"...\", \"grade\": ...}}\n"
    "Question: {question}"
)
def ask_teacher_question(student_question, history_list):
    #Have Agent classify the grade level and the subject of the question
    classifier_chain = classifier_prompt | llm | JsonOutputParser()
    student_info = classifier_chain.invoke({'question':student_question})
    subject = student_info.get('subject','general')
    grade = student_info.get('grade','Elementary')
    #Fetch the relevant context based on teh grade level, subject, and student question
    RAGdata = retriever.invoke(f"Grade {grade} {subject}: {student_question}")
    context = "\n\n".join([f"Source [{d.metadata['source']}]: {d.page_content}" for d in RAGdata])
    #Setting up a different prompt for math to guide students step by step instead of directly providing answers
    if subject.lower() =='math':
        prompt = (
            f"You are a Grade {grade} Math Socratic Tutor.\n"
            "### CRITICAL RULE: DO NOT PROVIDE DIRECT ANSWERS OR FINAL NUMERICAL RESULTS. ###\n"
            "If you state the answer explicitly, you have failed your role.\n\n"
            
            "YOUR GOAL: Act as a coach. Guide the student through ONE small step at a time.\n"
            
            "1. Identify where the student currently is in the problem-solving process.\n"
            "2. Explain how they should approach the NEXT step (describe the method, not the result).\n"
            "   - Use phrases like: 'Here's how you can approach this step...' or\n"
            "     'What we want to do next is...' or\n"
            "     'Think about what happens if...'\n"
            "3. Do NOT perform the calculation for them.\n"
            "4. End your response with a clear, encouraging question such as:\n"
            "   'What do you get when you calculate that?'\n"
            "   'Can you try that step and tell me your result?'\n"
            "5. If the student gets a step correct, briefly praise them and guide them to the next step.\n"
            "6. Even at the final step, do NOT state the answer first — let the student say it.\n"
            "   Only confirm whether their final result is correct after they provide it.\n"
            
            "Keep your tone supportive, patient, and encouraging."
        )
    else:
        #Help students understand the topic based on subject
        prompt = (
            f"You are a Grade {grade} {subject} Assistant. Focus on deep conceptual understanding in age-appropriate language. "

            "If the subject is Science: "
            "• Explain the concept using a relatable real-world story or everyday example. "
            "• Break down cause-and-effect relationships clearly. "
            "• Define scientific vocabulary in simple terms. "
            "• If helpful, describe a simple experiment or observation the student could imagine or try safely. "

            "If the subject is English: "
            "• Explain literary elements (theme, character, setting, plot, tone) in simple language. "
            "• Define difficult vocabulary words and use them in new example sentences. "
            "• If analyzing a text, explain why the author may have made certain choices. "
            "• Provide a short model example when helpful (e.g., a sample sentence or paragraph). "

            "If the subject is Social Studies: "
            "• Explain historical events, geography, or civics concepts using a story-based approach. "
            "• Clarify timelines, cause-and-effect, and why the topic matters today. "
            "• Define important terms (e.g., democracy, economy, culture) in simple language. "
            "• Connect the topic to real-life modern examples when possible. "

            "For all subjects: "
            "1. Explain the concept using a relatable story. "
            "2. Define any tricky vocabulary words found in the text. "
            "3. Give a clear final summary answer in 2–3 sentences. "
            "4. Suggest 3 'Think about it' discussion questions that encourage critical thinking."
        )
    #Have the agent to answer the question based on the prompt and context
    question_prompt = ChatPromptTemplate.from_messages([
            ("system", f"{prompt}\n\nTEXTBOOK CONTEXT:\n{context}"),
            MessagesPlaceholder(variable_name="history"),
            ("human", "{question}"),
        ])
    chain = question_prompt | llm | StrOutputParser()
    response = chain.invoke({"question": student_question, "history": history_list})
    #Store the results in chat history
    history_list.append(HumanMessage(content=student_question))
    history_list.append(AIMessage(content=response))
    
    return {
        "detected_subject": subject,
        "detected_grade": grade,
        "response": response
    }

print(ask_teacher_question("How do I add 1/2 and 1/4?",chat_history))
print(ask_teacher_question("What is a Pronoun",chat_history))

{'detected_subject': 'Math', 'detected_grade': 4, 'response': "Great question! Adding fractions is an important skill. Let's start by thinking about what these fractions represent.\n\nWe have 1/2 and 1/4. The first step in adding fractions is to check if they have the same denominator (the bottom number). Do 1/2 and 1/4 have the same denominator?"}
{'detected_subject': 'English', 'detected_grade': 3, 'response': 'Excellent question! Let\'s learn about pronouns with a story.\n\n### **A Story About Pronouns**\n\nImagine you have a friend named **Leo**. Leo loves to play soccer. Now, imagine you\'re telling me about Leo\'s day.\n\n*   Without pronouns, you might say: "**Leo** woke up early. Then **Leo** ate breakfast. After that, **Leo** put on **Leo\'s** cleats and went to **Leo\'s** soccer game."\n*   That sounds very repetitive, doesn\'t it? We keep saying "Leo" and "Leo\'s" over and over.\n\nA **pronoun** is a special word that **stands in for a noun** (like a person\'s name, a place,